In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys

sys.path.append("../")

In [ ]:


import pandas as pd

from src.data.base_datamodule import BaseDataModule
from src.data.butterfly_caption_builder import ButterflyCaptionBuilder
from src.data.butterfly_dataset import ButterflyDataset

In [ ]:
# BD = ButterflyDataset(
#     modalities={"coords": None}, data_dir=os.path.join(os.environ["DATA_DIR"]), use_aux_data='all'
# )

BD = ButterflyDataset(
    modalities={"coords": None},
    data_dir=os.path.join(os.environ["DATA_DIR"]),
    use_aux_data="all",
    use_target_data=False,
    use_unlabelled_data=True,
)

BCB = ButterflyCaptionBuilder(
    data_dir=os.path.join(os.environ["DATA_DIR"], "s2bms"),
    templates_fname="v6.json",
    concepts_fname="v4.json",
    seed=42,
)

BM = BaseDataModule(
    dataset=BD,
    batch_size=32,
    split_mode="from_file",
    saved_split_file_name=os.path.join(
        os.environ["DATA_DIR"],
        "s2bms",
        "splits/s2bms_unlabelled_union_val_test.pth",
    ),
    caption_builder=BCB,
)

In [ ]:
for dp in BD:
    assert len(dp["target"]) == 0
    assert len(dp["eo"]) == 1
    assert len(dp["eo"]["coords"]) == 2 and not any(dp["eo"]["coords"].isnan())
    assert not any(dp["aux"]["aux"].isnan())
    # assert all([type(x) == str for x in dp["aux"]["top"]])

In [ ]:
BM.setup()
len(BM.data_train), len(BM.data_val), len(BM.data_test)

In [ ]:
# BM.split_dir
split_files = os.listdir(BM.split_dir)
split_indices_joint = {}

for ifile, filename in enumerate(split_files):
    if filename.endswith(".pth"):
        filepath = os.path.join(BM.split_dir, filename)
        split_indices = BM.load_split_indices(filepath=filepath)
        if "clusters" in split_indices.keys():
            del split_indices[
                "clusters"
            ]  # remove clusters if present, as they are not relevant for the joint split
        if ifile == 0:
            split_indices_joint = split_indices
        else:  # concatenate series and reset index per key
            for key in split_indices_joint.keys():
                split_indices_joint[key] = pd.concat(
                    [split_indices_joint[key], split_indices[key]], ignore_index=True
                )
for k, v in split_indices_joint.items():
    print(f"{k}: {len(v)} samples")
    split_indices_joint[k] = v.reset_index(drop=True)

BM.save_split_indices(split_indices_joint)

In [ ]:
# batch = next(iter(BM.train_dataloader()))
for batch in BM.train_dataloader():
    break
# batch = next(iter(BM.val_dataloader()))
batch.keys()

In [ ]:
batch["eo"].keys()

In [ ]:
BCB.column_to_metadata_map["aux"]["aux_corine_frac_243"]

In [ ]:
template_id = 5
data_id_in_batch = 5

cap_num = BCB._build_from_template(
    template_idx=template_id,
    aux=batch["aux"]["aux"][data_id_in_batch, :],
    top=batch["aux"]["top"][data_id_in_batch],
    convert_corine_perc=False,
)
cap_adj = BCB._build_from_template(
    template_idx=template_id,
    aux=batch["aux"]["aux"][data_id_in_batch, :],
    top=batch["aux"]["top"][data_id_in_batch],
    convert_corine_perc=True,
)

print(cap_num)
print(cap_adj)

"setting with a mosaic of <aux_corine_frac_lowlevel_top_1>, <aux_corine_frac_lowlevel_top_2> and <aux_corine_frac_lowlevel_top_3>, with <aux_bioclim_12>, <aux_pop_density> and <aux_bioclim_08>.",


In [ ]:
batch["text"]